# RAG Types

**Module:** 04 — RAG

From Basic RAG to Graph, Agentic, Self-RAG, and CRAG—when each pattern earns its complexity.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Contrast basic vs modular RAG architectures
- Explain Graph RAG and relation-heavy questions
- Describe agentic retrieval loops and budgets
- Apply Self-RAG and CRAG control flows conceptually


## Basic RAG

**Definition.** Single-shot retrieve-then-generate over chunk embeddings.

**Why it matters.** Simplest path to grounded QA; baseline for all upgrades.

**How it works.** Index chunks; query embed; top-k; prompt; answer.

**Intuition.** One search, one answer.

**Common pitfalls.**
- Assuming basic RAG solves multi-hop
- No eval baseline

**When to use.** FAQ/policy QA with mostly single-hop questions.

### Type comparison

| Type | Complexity | Strength | Weakness |
|------|------------|----------|----------|
| Basic | Low | Fast baseline | Multi-hop weak |
| Modular | Med | Flexible | Ops complexity |
| Graph | High | Relations | Build cost |
| Agentic | High | Hard tasks | Cost/loops |
| Self-RAG | Med-High | Adaptive | Training/prompting |
| CRAG | Med | Noise robust | Cascades |
```mermaid
flowchart TB
  B[Basic] --> M[Modular]
  M --> G[Graph]
  M --> A[Agentic]
  M --> S[Self-RAG]
  M --> C[CRAG]
```


In [ ]:
# Demo 1 — Basic RAG loop
def basic_rag(query, index, llm):
    hits = index.search(query, k=4)
    return llm.generate(query, hits)
class Index:
    def search(self, q, k=4): return [f"chunk about {q}"][:k]
class LLM:
    def generate(self, q, hits): return f"Answer to {q} using {len(hits)} chunks"
print(basic_rag("refunds", Index(), LLM()))


In [ ]:
# Demo 2 — baseline metrics dict
baseline = {"recall@5": 0.62, "faithfulness": 0.71, "latency_ms_p50": 900}
print(baseline)
print("Improve modules only if they beat this baseline on a fixed eval set.")


In [ ]:
# Demo 3 — API shape
import json
print(json.dumps({"query": "refunds?", "top_k": 4, "answer": "...", "citations": ["C1"]}, indent=2))


### Try it yourself — Basic RAG

1. Give one production example that needs Basic RAG.
2. List two risks unique to Basic RAG.
3. State what you would measure before adopting it over Basic RAG.


## Advanced / Modular RAG

**Definition.** Composable graph of query transforms, multiple retrievers, rerankers, routers.

**Why it matters.** Real systems need branching, fallbacks, and per-intent paths.

**How it works.** Treat each step as a module with clear I/O; orchestrate with conditions.

**Intuition.** LEGO pipeline instead of a single script.

**Common pitfalls.**
- Over-modularizing before a baseline works
- Hidden state between modules

**When to use.** When basic RAG plateaus on eval.


In [ ]:
# Demo 1 — control-flow sketch for Advanced / Modular RAG
STEPS = {
  "Basic RAG": ["retrieve", "generate"],
  "Advanced / Modular RAG": ["route", "rewrite", "retrieve", "rerank", "generate"],
  "Graph RAG": ["entity_link", "subgraph", "verbalize", "generate"],
  "Agentic RAG": ["plan", "tool_search", "observe", "loop_or_answer"],
  "Self-RAG": ["decide_retrieve", "generate", "critique", "revise"],
  "CRAG (Corrective RAG)": ["retrieve", "grade", "correct_if_needed", "generate"],
}["Advanced / Modular RAG"]
state = "start"
for s in STEPS:
    print(f"{state} -> {s}"); state = s
print(state, "-> done")


In [ ]:
# Demo 2 — when to escalate from Basic RAG to Advanced / Modular RAG
def choose(pattern: str) -> str:
    if pattern == "single_hop_faq": return "Basic RAG"
    if pattern == "multi_hop_relations": return "Graph RAG"
    if pattern == "noisy_retrieval": return "CRAG (Corrective RAG)"
    if pattern == "multi_tool_research": return "Agentic RAG"
    if pattern == "adaptive_retrieve": return "Self-RAG"
    return "Advanced / Modular RAG"
print("this notebook section prefers:", "Advanced / Modular RAG")
for p in ["single_hop_faq", "multi_hop_relations", "noisy_retrieval", "multi_tool_research"]:
    print(p, "->", choose(p))


### Try it yourself — Advanced / Modular RAG

1. Give one production example that needs Advanced / Modular RAG.
2. List two risks unique to Advanced / Modular RAG.
3. State what you would measure before adopting it over Basic RAG.


## Graph RAG

**Definition.** Retrieval over entities/relations (knowledge graph) plus or instead of chunks.

**Why it matters.** Multi-hop and relationship questions need structure, not only similar text.

**How it works.** Extract entities/edges; retrieve subgraphs; verbalize into LLM context.

**Intuition.** Map + encyclopedia instead of only sticky notes.

**Common pitfalls.**
- Graph construction cost
- Noisy extraction

**When to use.** Org charts, dependencies, regulatory relationships.


In [ ]:
# Demo 1 — tiny knowledge graph retrieve
from collections import defaultdict
edges = [("Acme", "owns", "ShopApp"), ("ShopApp", "policy", "Refund60Days"),
         ("Refund60Days", "states", "60 days")]
spo = defaultdict(list)
for s, p, o in edges: spo[s].append((p, o))

def subgraph(entity, depth=2):
    seen={entity}; frontier=[entity]; triples=[]
    for _ in range(depth):
        nxt=[]
        for e in frontier:
            for p,o in spo.get(e, []):
                triples.append((e,p,o)); 
                if o not in seen: seen.add(o); nxt.append(o)
        frontier=nxt
    return triples
print(subgraph("Acme"))


In [ ]:
# Demo 2 — verbalize triples for LLM context
triples = subgraph("Acme")
ctx = "; ".join(f"({s})-[{p}]->({o})" for s,p,o in triples)
print("CONTEXT:", ctx)


### Try it yourself — Graph RAG

1. Give one production example that needs Graph RAG.
2. List two risks unique to Graph RAG.
3. State what you would measure before adopting it over Basic RAG.


## Agentic RAG

**Definition.** An agent decides when/what to retrieve, may call tools iteratively.

**Why it matters.** Complex tasks need planning, multiple searches, and verification loops.

**How it works.** Planner selects tools (search, SQL, web); observes; continues until budget.

**Intuition.** Research intern who can run multiple searches before writing.

**Common pitfalls.**
- Unbounded loops/cost
- Tool errors ignored

**When to use.** Analyst copilots and multi-source research.


In [ ]:
# Demo 1 — agentic loop with budget
def agentic_rag(question, tools, max_steps=3):
    notes = []
    for step in range(max_steps):
        action = "search" if step == 0 else "answer"
        if action == "search":
            notes.append(tools["search"](question)); continue
        return {"answer": f"Draft from {notes}", "steps": step+1}
    return {"answer": "budget exceeded", "steps": max_steps}
tools = {"search": lambda q: f"hits for {q}"}
print(agentic_rag("compare refund vs exchange", tools))


In [ ]:
# Demo 2 — tool call JSON shape
import json
print(json.dumps({
  "tool": "vector_search",
  "arguments": {"query": "refund policy", "tenant": "acme", "k": 8},
  "observation": {"hits": [{"id": "C1", "score": 0.81}]},
}, indent=2))


### Try it yourself — Agentic RAG

1. Give one production example that needs Agentic RAG.
2. List two risks unique to Agentic RAG.
3. State what you would measure before adopting it over Basic RAG.


## Self-RAG

**Definition.** Model critiques whether to retrieve and whether evidence supports the claim.

**Why it matters.** Adaptive retrieval can skip search when unnecessary and reject bad evidence.

**How it works.** Special tokens/scores for retrieve/grade/critique during generation.

**Intuition.** Think → fetch? → write → fact-check.

**Common pitfalls.**
- Needs specialized training or careful prompting
- Latency

**When to use.** When always-on retrieval wastes tokens or adds noise.


In [ ]:
# Demo 1 — self-RAG style gates
def self_rag_gates(query, confidence_need_retrieval, evidence_score):
    retrieve = confidence_need_retrieval > 0.5
    accept = evidence_score > 0.6
    return {"retrieve": retrieve, "accept_evidence": accept,
            "action": "generate" if (not retrieve or accept) else "re_retrieve"}
print(self_rag_gates("hi", 0.1, 0.0))
print(self_rag_gates("refund law?", 0.9, 0.2))
print(self_rag_gates("refund law?", 0.9, 0.85))


In [ ]:
# Demo 2 — critique checklist
critiques = ["supported", "unsupported", "ambiguous"]
for c in critiques:
    print(c, "->", "keep" if c=="supported" else "revise/retrieve")


### Try it yourself — Self-RAG

1. Give one production example that needs Self-RAG.
2. List two risks unique to Self-RAG.
3. State what you would measure before adopting it over Basic RAG.


## CRAG (Corrective RAG)

**Definition.** Evaluate retrieved docs; if weak, trigger corrective actions (web, rewrite, expand).

**Why it matters.** Prevents confidently answering from irrelevant top-k.

**How it works.** Retrieval evaluator → if fail, corrective retrieval → generate.

**Intuition.** If the binder is wrong, get a better binder before writing.

**Common pitfalls.**
- Evaluator false positives/negatives
- Costly cascades

**When to use.** Noisy corpora and open-domain settings.


In [ ]:
# Demo 1 — corrective evaluator
def grade_hits(hits, min_score=0.55):
    best = max((h["score"] for h in hits), default=0)
    return best >= min_score, best
hits = [{"id": "C1", "score": 0.42}, {"id": "C2", "score": 0.39}]
ok, best = grade_hits(hits)
print("accept" if ok else "corrective path", "best=", best)


In [ ]:
# Demo 2 — corrective actions menu
def correct(query):
    return ["rewrite_query", "hybrid_search", "web_fallback"]
print(correct("refund"))


In [ ]:
# Demo 3 — full CRAG-ish control flow
def crag(query, retrieve, grade, correct, generate):
    hits = retrieve(query)
    ok, _ = grade(hits)
    if not ok:
        query = query + " policy"; hits = correct(query)
    return generate(query, hits)
print(crag("refund", lambda q: [{"score": 0.2}], grade_hits,
           lambda q: [{"score": 0.9, "id": "C9"}],
           lambda q, h: f"ans using {h}"))


### Try it yourself — CRAG (Corrective RAG)

1. Give one production example that needs CRAG (Corrective RAG).
2. List two risks unique to CRAG (Corrective RAG).
3. State what you would measure before adopting it over Basic RAG.


## Glossary

- **HyDE**: Hypothetical document embeddings for query expansion
- **CRAG**: Corrective RAG with retrieval evaluation


## Summary & Key Takeaways

- Start basic; earn complexity with eval gains
- Graph helps relationships; agents help multi-step research
- Self-RAG adapts retrieval; CRAG corrects weak evidence
- Always budget latency and tool calls

### Practice

Map five real user questions to the simplest RAG type that could work.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
